# Shelter Allocation Simulation

## Objective

This notebook allocates affected people from each disaster event to the nearest available temporary shelters.

Unlike a simple nearest-neighbor assignment, this notebook performs **capacity-aware multi-shelter allocation**, where:

- Every shelter has a finite accommodation capacity.
- Multiple shelters can be assigned for a single disaster.
- Shelter capacities decrease after allocation.
- Later disaster events use the remaining shelter capacities.
- Allocation continues until either:
    - all affected people are accommodated, or
    - no nearby shelter capacity remains.

This notebook simulates the temporary accommodation planning performed by disaster management authorities.

---

## Inputs

- final_disaster_dataset_engineered.csv
- affected_population.csv
- shelter_infrastructure.parquet

---

## Output

shelter_allocation.csv

Each disaster event will contain:

- Assigned shelters
- Number of shelters used
- Total allocated capacity
- Remaining unallocated people
- Average shelter distance
- Allocation status

This output will later be used by:

- Transportation Planning
- Rescue Team Deployment
- Final Disaster Response Simulation

In [60]:
# ============================================================
# Imports
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.neighbors import KDTree

from shapely.geometry import Point

from tqdm.auto import tqdm

In [61]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "datasets"

OUTPUT_DIR = DATA_DIR

print(PROJECT_ROOT)

C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination


In [62]:
# ============================================================
# Load Datasets
# ============================================================

events = pd.read_csv(

    DATA_DIR / "affected_population.csv"

)

shelters = gpd.read_parquet(

    DATA_DIR / "shelter_infrastructure.parquet"

)

print()

print("Disaster Events")

print(events.shape)

print()

print("Shelter Dataset")

print(shelters.shape)


Disaster Events
(2374, 32)

Shelter Dataset
(27537, 63)


In [63]:
# ============================================================
# Dataset Verification
# ============================================================

required_event_columns = [

    "SourceEventID",

    "Latitude",

    "Longitude",

    "AffectedPopulation"

]

required_shelter_columns = [

    "Latitude",

    "Longitude",

    "EstimatedCapacity"

]

missing_events = [

    c

    for c in required_event_columns

    if c not in events.columns

]

missing_shelters = [

    c

    for c in required_shelter_columns

    if c not in shelters.columns

]

assert len(missing_events) == 0, missing_events

assert len(missing_shelters) == 0, missing_shelters

print()

print("All required columns found.")


All required columns found.


In [64]:
# ============================================================
# Shelter Inventory
# ============================================================

shelters = shelters.copy()

shelters["RemainingCapacity"] = (

    shelters["EstimatedCapacity"]

    .astype(int)

)

shelters = shelters[

    shelters["RemainingCapacity"] > 0

]

shelters.reset_index(

    drop=True,

    inplace=True

)

print()

print("Available Shelters")

print(len(shelters))

print()

print(

    shelters["RemainingCapacity"]

    .describe()

)


Available Shelters
27537

count    2.753700e+04
mean     1.277289e+04
std      1.029206e+05
min      5.000000e+01
25%      6.000000e+02
50%      8.710000e+02
75%      4.124000e+03
max      7.910573e+06
Name: RemainingCapacity, dtype: float64


In [65]:
# ============================================================
# Build KDTree
# ============================================================

EARTH_KM_PER_DEGREE = 111.32

shelter_coordinates = np.column_stack(

    (

        shelters["Latitude"],

        shelters["Longitude"]

    )

)

tree = KDTree(

    shelter_coordinates,

    metric="euclidean"

)

print()

print("KDTree Built Successfully")

print()

print("Shelters Indexed :", len(shelters))


KDTree Built Successfully

Shelters Indexed : 27537


In [66]:
# ============================================================
# Haversine Distance
# ============================================================

def haversine_distance(

    lat1,

    lon1,

    lat2,

    lon2

):

    R = 6371.0

    lat1 = np.radians(lat1)

    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)

    lon2 = np.radians(lon2)

    dlat = lat2 - lat1

    dlon = lon2 - lon1

    a = (

        np.sin(dlat / 2) ** 2

        +

        np.cos(lat1)

        *

        np.cos(lat2)

        *

        np.sin(dlon / 2) ** 2

    )

    c = 2 * np.arctan2(

        np.sqrt(a),

        np.sqrt(1 - a)

    )

    return R * c

In [67]:
# ============================================================
# Allocate Shelters
# ============================================================

def allocate_shelters(

    event_lat,

    event_lon,

    affected_population,

    k=100

):

    distances, indices = tree.query(

        [[event_lat, event_lon]],

        k=min(

            k,

            len(shelters)

        )

    )

    remaining = int(affected_population)

    allocated_capacity = 0

    shelter_ids = []

    shelter_names = []

    shelter_distances = []

    used_shelters = 0

    for idx in indices[0]:

        if remaining <= 0:

            break

        capacity = shelters.loc[

            idx,

            "RemainingCapacity"

        ]

        if capacity <= 0:

            continue

        allocation = min(

            remaining,

            capacity

        )

        shelters.loc[

            idx,

            "RemainingCapacity"

        ] -= allocation

        remaining -= allocation

        allocated_capacity += allocation

        used_shelters += 1

        shelter_ids.append(

            int(

                shelters.loc[idx, "id"]

            )

        )

        if "name" in shelters.columns:

            shelter_names.append(

                str(

                    shelters.loc[idx, "name"]

                )

            )

        else:

            shelter_names.append(

                "Unnamed Shelter"

            )

        distance = haversine_distance(

            event_lat,

            event_lon,

            shelters.loc[idx, "Latitude"],

            shelters.loc[idx, "Longitude"]

        )

        shelter_distances.append(

            round(

                distance,

                2

            )

        )

    if remaining == 0:

        status = "Fully Allocated"

    elif allocated_capacity > 0:

        status = "Partially Allocated"

    else:

        status = "No Shelter Available"

    return {

        "AssignedShelterIDs": shelter_ids,

        "AssignedShelterNames": shelter_names,

        "ShelterDistancesKm": shelter_distances,

        "SheltersUsed": used_shelters,

        "AllocatedCapacity": allocated_capacity,

        "RemainingPeople": remaining,

        "AllocationStatus": status

    }

In [68]:
# ============================================================
# Test Allocation
# ============================================================

sample = events.iloc[0]

result = allocate_shelters(

    sample["Latitude"],

    sample["Longitude"],

    sample["AffectedPopulation"]

)

print()

for k, v in result.items():

    print(

        f"{k} :",

        v

    )


AssignedShelterIDs : [662890468]
AssignedShelterNames : ['Lopon Stadium']
ShelterDistancesKm : [np.float64(17.61)]
SheltersUsed : 1
AllocatedCapacity : 3374
RemainingPeople : 0
AllocationStatus : Fully Allocated


In [69]:
# ============================================================
# Allocate Shelters for All Disaster Events
# ============================================================

events = events.sort_values(

    by=[

        "ImpactScore",

        "AffectedPopulation"

    ],

    ascending=False

).reset_index(drop=True)

allocation_results = []

print()

print("Allocating Shelters...")

for _, event in tqdm(

    events.iterrows(),

    total=len(events)

):

    allocation = allocate_shelters(

        event["Latitude"],

        event["Longitude"],

        event["AffectedPopulation"]

    )

    allocation_results.append(allocation)

print()

print("Shelter Allocation Completed")


Allocating Shelters...


  0%|          | 0/2374 [00:00<?, ?it/s]


Shelter Allocation Completed


In [70]:
# ============================================================
# Convert Allocation Results
# ============================================================

allocation_df = pd.DataFrame(

    allocation_results

)

print()

print(allocation_df.shape)

allocation_df.head()


(2374, 7)


,AssignedShelterIDs,AssignedShelterNames,ShelterDistancesKm,SheltersUsed,AllocatedCapacity,RemainingPeople,AllocationStatus
0,"[140960345, 140960342, 140960353, 424725846, 1...","[SARASWATI SCHOOL, None, None, DPS Gandhidham,...","[39.1, 39.35, 39.36, 39.59, 39.81, 39.83, 39.8...",22,182813,0,Fully Allocated
1,"[7030363185, 7030363285, 13948164995, 66477600...","[None, None, National Sports University Koutru...","[1.18, 4.74, 21.05, 22.46, 25.86, 24.27, 26.54...",11,204962,0,Fully Allocated
2,"[6098807145, 6098807832, 6098807132, 609880732...","[Saint Ursula High School, Don Bosco, None, No...","[9.72, 12.13, 26.0, 27.84, 27.93, 28.61, 31.13...",40,382680,0,Fully Allocated
3,"[1176703629, 1381550621]","[KGBV Kendriya Vidyalaya, Govt Boys Inter Coll...","[46.93, 50.67]",2,1217,0,Fully Allocated
4,"[1254857478, 11507984405, 11508770835, 1150801...","[None, Sister Margaret Nobel School, Higher Se...","[8.68, 10.91, 11.15, 11.55, 12.02, 12.11, 12.3...",70,433001,0,Fully Allocated


In [71]:
# ============================================================
# Merge Allocation Results
# ============================================================

events = pd.concat(

    [

        events.reset_index(drop=True),

        allocation_df

    ],

    axis=1

)

print()

print(events.shape)

events.head()


(2374, 39)


,SourceEventID,DisasterType,Latitude,Longitude,Magnitude,Depth,Time,Temperature,Humidity,Rainfall,...,Adults,Elderly,VulnerabilityScore,AssignedShelterIDs,AssignedShelterNames,ShelterDistancesKm,SheltersUsed,AllocatedCapacity,RemainingPeople,AllocationStatus
0,18251,Earthquake,23.4190,70.2320,7.7,16.0,2001-01-26 03:16:40.500000,0.587302,0.442105,0.000000,...,109688,23766,1.12,"[140960345, 140960342, 140960353, 424725846, 1...","[SARASWATI SCHOOL, None, None, DPS Gandhidham,...","[39.1, 39.35, 39.36, 39.59, 39.81, 39.83, 39.8...",22,182813,0,Fully Allocated
1,5900,Earthquake,24.8036,93.6505,6.7,55.0,2016-01-03 23:05:22.270000,0.611111,0.884211,0.000000,...,122977,26645,1.12,"[7030363185, 7030363285, 13948164995, 66477600...","[None, None, National Sports University Koutru...","[1.18, 4.74, 21.05, 22.46, 25.86, 24.27, 26.54...",11,204962,0,Fully Allocated
2,3158,Earthquake,26.7813,92.4565,6.0,34.0,2021-04-28 02:21:26.212000,0.812169,0.557895,0.008197,...,229608,49748,1.12,"[6098807145, 6098807832, 6098807132, 609880732...","[Saint Ursula High School, Don Bosco, None, No...","[9.72, 12.13, 26.0, 27.84, 27.93, 28.61, 31.13...",40,382680,0,Fully Allocated
3,13493,Earthquake,34.5390,73.5880,7.6,26.0,2005-10-08 03:50:40.800000,0.670635,0.347368,0.000000,...,730,158,1.12,"[1176703629, 1381550621]","[KGBV Kendriya Vidyalaya, Govt Boys Inter Coll...","[46.93, 50.67]",2,1217,0,Fully Allocated
4,4603,Earthquake,26.3711,90.1611,5.3,10.0,2018-09-12 04:50:46.570000,0.775132,0.968421,0.877049,...,259801,56290,1.12,"[1254857478, 11507984405, 11508770835, 1150801...","[None, Sister Margaret Nobel School, Higher Se...","[8.68, 10.91, 11.15, 11.55, 12.02, 12.11, 12.3...",70,433001,0,Fully Allocated


In [72]:
# ============================================================
# Allocation Statistics
# ============================================================

events["AverageShelterDistanceKm"] = (

    events["ShelterDistancesKm"]

    .apply(

        lambda x:

        round(np.mean(x),2)

        if len(x)>0

        else np.nan

    )

)

events["MaximumShelterDistanceKm"] = (

    events["ShelterDistancesKm"]

    .apply(

        lambda x:

        round(np.max(x),2)

        if len(x)>0

        else np.nan

    )

)

events["MinimumShelterDistanceKm"] = (

    events["ShelterDistancesKm"]

    .apply(

        lambda x:

        round(np.min(x),2)

        if len(x)>0

        else np.nan

    )

)

In [73]:
# ============================================================
# Human Readable Numbers
# ============================================================

def human_number(value):

    value = int(value)

    if value >= 1_00_00_000:
        return f"{value:,} ({value/1_00_00_000:.2f} Crore)"

    elif value >= 1_00_000:
        return f"{value:,} ({value/1_00_000:.2f} Lakh)"

    elif value >= 1000:
        return f"{value:,} ({value/1000:.2f} Thousand)"

    return str(value)

In [83]:
total_events = len(events)

full = (events["AllocationStatus"] == "Fully Allocated").sum()

partial = (events["AllocationStatus"] == "Partially Allocated").sum()

none = (events["AllocationStatus"] == "No Shelter Available").sum()

print()

print()

print("="*75)
print("Shelter Allocation Summary")
print("="*75)

print()

print(f"Total Disaster Events Processed : {total_events}")

print(
    f"Fully Allocated : {full}"
    "  -> Every affected person received temporary shelter."
)

print(
    f"Partially Allocated : {partial}"
    "  -> Some people received shelter, but nearby shelter capacity became insufficient."
)

print(
    f"No Shelter Available : {none}"
    "  -> No remaining shelter capacity could be assigned for these events."
)


print(
    f"\nAverage Distance Travelled to Assigned Shelter : "
    f"{events['AverageShelterDistanceKm'].mean():.2f} km"
)

print()

waiting = events["RemainingPeople"].sum()

print()

print(
    "People Still Waiting For Temporary Shelter :"
)

print(human_number(waiting))



Shelter Allocation Summary

Total Disaster Events Processed : 2374
Fully Allocated : 762  -> Every affected person received temporary shelter.
Partially Allocated : 128  -> Some people received shelter, but nearby shelter capacity became insufficient.
No Shelter Available : 1484  -> No remaining shelter capacity could be assigned for these events.

Average Distance Travelled to Assigned Shelter : 186.01 km


People Still Waiting For Temporary Shelter :
134,941,587 (13.49 Crore)


In [75]:
# ============================================================
# Convert List Columns to Strings
# ============================================================

list_columns = [

    "AssignedShelterIDs",

    "AssignedShelterNames",

    "ShelterDistancesKm"

]

for col in list_columns:

    events[col] = events[col].apply(

        lambda x: ";".join(map(str, x))

        if isinstance(x, list)

        else ""

    )

In [84]:
initial_capacity = shelters["EstimatedCapacity"].sum()

remaining_capacity = shelters["RemainingCapacity"].sum()

used_capacity = initial_capacity - remaining_capacity

print()

print("="*75)

print("Shelter Capacity Summary")

print("="*75)

print()

print("Initial Shelter Capacity")

print(human_number(initial_capacity))

print()

print("Capacity Used")

print(human_number(used_capacity))

print()

print("Remaining Capacity")

print(human_number(remaining_capacity))


Shelter Capacity Summary

Initial Shelter Capacity
351,726,982 (35.17 Crore)

Capacity Used
61,412,556 (6.14 Crore)

Remaining Capacity
290,314,426 (29.03 Crore)


In [88]:
# ============================================================
# Remaining Shelter Capacity Summary
# ============================================================

print()

print("=" * 75)

print("Remaining Shelter Capacity Summary")

print("=" * 75)

initial_capacity = shelters["EstimatedCapacity"].sum()

remaining_capacity = shelters["RemainingCapacity"].sum()

used_capacity = initial_capacity - remaining_capacity

print()

print(f"Total Shelters Available              : {len(shelters):,}")

print(f"Fully Occupied Shelters               : {empty:,}")

print(f"Shelters Still Available              : {available:,}")

print()

print(f"Initial Shelter Capacity              : {human_number(initial_capacity)}")

print(f"Capacity Allocated During Simulation  : {human_number(used_capacity)}")

print(f"Remaining Shelter Capacity            : {human_number(remaining_capacity)}")

print()

print(f"Average Remaining Capacity/Shelter    : {shelters['RemainingCapacity'].mean():.0f} people")

print(f"Maximum Remaining Capacity            : {human_number(shelters['RemainingCapacity'].max())}")

print(f"Minimum Remaining Capacity            : {human_number(shelters['RemainingCapacity'].min())}")


Remaining Shelter Capacity Summary

Total Shelters Available              : 27,537
Fully Occupied Shelters               : 7,184
Shelters Still Available              : 20,353

Initial Shelter Capacity              : 351,726,982 (35.17 Crore)
Capacity Allocated During Simulation  : 61,412,556 (6.14 Crore)
Remaining Shelter Capacity            : 290,314,426 (29.03 Crore)

Average Remaining Capacity/Shelter    : 10543 people
Maximum Remaining Capacity            : 7,910,573 (79.11 Lakh)
Minimum Remaining Capacity            : 0


In [90]:
utilization = (

    used_capacity

    /

    initial_capacity

) * 100

print()

print(f"Shelter Capacity Utilization          : {utilization:.2f}%")


Shelter Capacity Utilization          : 17.46%


In [89]:
allocation_rate = (

    events["AllocatedCapacity"].sum()

    /

    events["AffectedPopulation"].sum()

) * 100

print()

print(f"Overall Allocation Efficiency : {allocation_rate:.2f}%")


Overall Allocation Efficiency : 31.28%


In [91]:
print()

print("="*75)
print("Allocation Constraints")
print("="*75)

print()

print("Nearest Shelters Considered Per Event :", 100)

print("Shelters Still Having Capacity :", available)

print("Remaining National Shelter Capacity :", human_number(remaining_capacity))

print()

print(
    "Note: Allocation is restricted to nearby shelters to simulate realistic disaster response."
)


Allocation Constraints

Nearest Shelters Considered Per Event : 100
Shelters Still Having Capacity : 20353
Remaining National Shelter Capacity : 290,314,426 (29.03 Crore)

Note: Allocation is restricted to nearby shelters to simulate realistic disaster response.


In [77]:
# ============================================================
# Save Shelter Allocation
# ============================================================

OUTPUT = (

    DATA_DIR

    / "shelter_allocation.csv"

)

events.to_csv(

    OUTPUT,

    index=False

)

print()

print("Saved Successfully")

print()

print(OUTPUT)


Saved Successfully

C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\shelter_allocation.csv


The final CSV will contain:
SourceEventID
Latitude
Longitude
AffectedPopulation
AssignedShelterIDs
AssignedShelterNames
ShelterDistancesKm
SheltersUsed
AllocatedCapacity
RemainingPeople
AllocationStatus
AverageShelterDistanceKm
MaximumShelterDistanceKm
MinimumShelterDistanceKm